# 9 — Decay and source fitting

**Theme:** extracting physical rates from a concentration peak.

When a source runs in a room and then stops, the concentration rises and then
decays away. Fitting that shape recovers quantities you cannot read off the
plot: how strong the source was, and how fast the room cleared it.

The decay combines ventilation and deposition to surfaces, so the fitted loss
rate is the *total* removal rate.

In [ ]:
import aerosoltools as at

elpi = at.load_elpi_file("../../tests/data/Sample_ELPI.txt")
fig, ax = elpi.plot_total_conc()
ax.set_title("An emission followed by a decay")

## Fitting a peak

`fit_decay` takes the time window containing the peak. Give it the period as a
`(start, end)` pair, or the name of a marked activity.

In [ ]:
result = elpi.fit_decay(
    period=("2023-09-07 09:07:50", "2023-09-07 09:11:00"),
    metric="PNC",
)

print("model    :", result["model"])
print("R squared:", round(result["r_squared"], 4))
print("points   :", result["n_points"])

`DecayResult` supports both attribute and key access, so either style works.

In [ ]:
print("background        :", round(result.background, 1), result.unit)
print("peak concentration:", round(result.peak_concentration, 1), result.unit)
print("peak excess       :", round(result.peak_excess, 1), result.unit)

## The rates

The decay rate is the physically interesting output. It is reported per second
and per hour, with the corresponding half-life.

In [ ]:
print(f"decay rate   : {result.decay_rate:.5f} per second")
print(f"             : {result.decay_rate_per_hour:.3f} per hour")
print(f"half-life    : {result.half_life_hours:.3f} hours "
      f"({result.half_life_hours * 60:.1f} minutes)")
print(f"decay R^2    : {result.decay_r_squared:.4f}")

## Source strength

To convert an observed rise into an emission *rate* you need the room volume:
a given source fills a small room faster than a large one. Supply `volume` in
cubic metres.

In [ ]:
with_volume = elpi.fit_decay(
    period=("2023-09-07 09:07:50", "2023-09-07 09:11:00"),
    metric="PNC",
    volume=30.0,
)

print(f"emission rate: {with_volume.emission_rate:.3e} "
      f"{with_volume.emission_rate_unit}")
print(f"emission duration: {with_volume.emission_duration_s:.0f} s")

## Separating ventilation from deposition

The fitted decay lumps together air exchange and losses to surfaces. If you know
the air exchange rate independently — from a tracer-gas measurement, or the
ventilation setpoint — pass it as `air_exchange_rate` (per hour) and the wall
loss is reported separately.

In [ ]:
split = elpi.fit_decay(
    period=("2023-09-07 09:07:50", "2023-09-07 09:11:00"),
    metric="PNC",
    volume=30.0,
    air_exchange_rate=0.5,
)

print(f"total decay  : {split.decay_rate_per_hour:.3f} per hour")
print(f"air exchange : 0.500 per hour (given)")
print(f"wall loss    : {split.loss_rate_per_hour:.3f} per hour")

## Choosing the window

The window should start before the rise and end while the decay is still
clean — before the concentration flattens into noise. Compare two windows:

In [ ]:
for start, end in [("2023-09-07 09:07:50", "2023-09-07 09:11:00"),
                   ("2023-09-07 09:07:50", "2023-09-07 09:09:30")]:
    fit = elpi.fit_decay(period=(start, end), metric="PNC")
    print(f"{start[-8:]} - {end[-8:]}  "
          f"decay {fit.decay_rate_per_hour:7.3f} /h   R2 {fit.r_squared:.4f}")

## Fitting a different metric

`metric` accepts anything `summarize_exposure` accepts, so a mass-based decay
is as easy as a number-based one — the mass fraction settles faster than the
number concentration, because it is dominated by larger particles.

In [ ]:
mass_fit = elpi.fit_decay(
    period=("2023-09-07 09:07:50", "2023-09-07 09:11:00"),
    metric="PM10",
)

print(f"PNC  decay: {result.decay_rate_per_hour:.3f} per hour")
print(f"PM10 decay: {mass_fit.decay_rate_per_hour:.3f} per hour")

## Fitting a marked activity

If the peak is already marked as an activity, pass its name instead of a time
pair.

In [ ]:
elpi.mark_activities({
    "Release": [("2023-09-07 09:07:50", "2023-09-07 09:11:00")],
})

by_name = elpi.fit_decay(period="Release", metric="PNC")
print(f"decay {by_name.decay_rate_per_hour:.3f} per hour, R2 {by_name.r_squared:.4f}")

---

**Next:** [10a — Correlation and agreement](10a-correlation-and-agreement.ipynb).